In [ ]:
import asyncio
from src.RAG.Strategies_RAG_Inference.query_expander import QueryExpander

async def test():
    expander = QueryExpander(model_name="qwen2.5:7b")

    query = "provide details on car loan"
    expanded = await expander.expand_query(query)

    return expanded  # ✅ return instead of just printing

In [4]:
query_batch = await test()
print(query_batch)

[2026-04-18 14:36:22,690] [INFO] TextGuardrails initialized successfully.
[2026-04-18 14:36:22,694] [INFO] QueryExpander active with qwen2.5:7b (Ollama native).
[2026-04-18 14:36:22,701] [INFO] apply completed in 0.00s
[2026-04-18 14:36:52,933] [INFO] apply completed in 0.02s
[2026-04-18 14:36:52,989] [INFO] Expanded into 1 queries.
[2026-04-18 14:36:52,993] [INFO] expand_query completed in 30.30s
['outline information regarding automobile financing']


In [6]:
import re

def normalize_section(section: str) -> str:
    if not section:
        return "default"

    section = section.lower().strip()
    section = re.sub(r"[^\w\s]", "", section)
    section = re.sub(r"\s+", " ", section)

    return section

In [7]:
import sys
from typing import Set, List

from sqlalchemy import select, func

from src.RAG.models import ChunkModel
from src.db.main import get_session
from src.Utils.logger_setup import setup_logger, current_logger, track_performance
from src.Utils.exception_handler import CustomException


logger = setup_logger("section_service")
current_logger.set(logger)


class SectionService:
    """
    Service to extract UNIQUE normalized section names
    from ChunkModel JSONB metadata.
    """

    @track_performance
    async def get_normalized_sections(self) -> List[str]:
        """
        Fetch unique section_name values from JSONB metadata,
        normalize them, and return sorted list.
        """

        try:
            sections: Set[str] = set()

            async for session in get_session():

                # ✅ DB-level DISTINCT extraction (fast)
                stmt = select(
                    func.distinct(
                        ChunkModel.chunk_metadata["section_name"].astext
                    )
                ).where(
                    ChunkModel.chunk_metadata["section_name"].isnot(None)
                )

                result = await session.execute(stmt)

                for row in result.scalars().all():
                    if not row:
                        continue

                    normalized = normalize_section(str(row).strip())
                    sections.add(normalized)

                break  # single session usage

            logger.info(f"Extracted {len(sections)} normalized sections.")
            return sorted(sections)

        except Exception as e:
            logger.error("Failed to extract normalized section names.")
            raise CustomException(e, sys)

In [8]:
sections_names = await SectionService().get_normalized_sections()

2026-04-18 13:42:01,026 INFO sqlalchemy.engine.Engine select pg_catalog.version()
2026-04-18 13:42:01,028 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-18 13:42:01,076 INFO sqlalchemy.engine.Engine select current_schema()
2026-04-18 13:42:01,078 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-18 13:42:01,084 INFO sqlalchemy.engine.Engine show standard_conforming_strings
2026-04-18 13:42:01,084 INFO sqlalchemy.engine.Engine [raw sql] ()
2026-04-18 13:42:01,098 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-18 13:42:01,116 INFO sqlalchemy.engine.Engine SELECT distinct(document_chunks.chunk_metadata ->> $1::TEXT) AS distinct_1 
FROM document_chunks 
WHERE document_chunks.chunk_metadata[$2::TEXT] IS NOT NULL
2026-04-18 13:42:01,120 INFO sqlalchemy.engine.Engine [generated in 0.00362s] ('section_name', 'section_name')
[2026-04-18 13:42:01,201] [INFO] Extracted 7 normalized sections.
[2026-04-18 13:42:01,201] [INFO] get_normalized_sections completed in 0.45s


2026-04-18 13:42:01,222 INFO sqlalchemy.engine.Engine ROLLBACK


In [9]:
sections_names

['drivesure auto loan dal',
 'edugrow education loan',
 'flexiserve personal loan fpl',
 'homesecure home loan hhl',
 'homesure mortgage loan hml',
 'securesave loan against approved securities slaas',
 'tradeplus business loan tbl']

In [10]:
import json
import numpy as np
import ollama
import asyncio



class QueryClassifier:

    def __init__(self, llm_model="qwen2.5:7b", embed_model="nomic-embed-text", threshold=0.55):
        self.llm_model = llm_model
        self.embed_model = embed_model
        self.threshold = threshold

        self.section_service = SectionService()

        self.section_list = []
        self.section_embeddings = None

    # ----------------------------
    # SINGLE EMBEDDING CALL (OLLAMA)
    # ----------------------------
    async def _embed(self, text: str):

        def _call():
            return ollama.embeddings(
                model=self.embed_model,
                prompt=f"search_document: {text}"
            )

        response = await asyncio.to_thread(_call)
        return response["embedding"]

    # ----------------------------
    # Load + embed DB sections
    # ----------------------------
    async def prepare_context(self):

        self.section_list = await self.section_service.get_normalized_sections()

        if not self.section_list:
            self.section_list = ["default"]

        vectors = []
        for s in self.section_list:
            emb = await self._embed(s)
            vectors.append(emb)

        vectors = np.array(vectors, dtype="float32")

        norms = np.linalg.norm(vectors, axis=1, keepdims=True) + 1e-9
        self.section_embeddings = vectors / norms

    # ----------------------------
    # cosine match helper
    # ----------------------------
    async def _cosine_match(self, text: str):

        vec = await self._embed(text)
        vec = np.array(vec, dtype="float32")
        vec = vec / (np.linalg.norm(vec) + 1e-9)

        sims = np.dot(self.section_embeddings, vec)

        top2_idx = np.argsort(sims)[-2:][::-1]

        results = []
        for idx in top2_idx:
            if sims[idx] >= self.threshold:
                results.append(self.section_list[idx])

        return results[:2] if results else self.section_list[:2]

    # ----------------------------
    # MAIN: batch classification
    # ----------------------------
    async def classify_batch(self, queries: list[str]) -> dict:

        if not self.section_list:
            await self.prepare_context()

        prompt = f"""
You are a classifier.

Available sections:
{self.section_list}

Return JSON like:
{{
  "query": ["section1", "section2"]
}}

Queries:
{queries}
"""

        response = ollama.chat(
            model=self.llm_model,
            messages=[{"role": "user", "content": prompt}],
            options={"temperature": 0}
        )

        content = response["message"]["content"]

        try:
            raw_output = json.loads(content)

            final_output = {}

            for query, sections in raw_output.items():

                normalized_sections = []

                for s in sections:
                    s = normalize_section(s)
                    matches = await self._cosine_match(s)
                    normalized_sections.extend(matches)

                final_output[query] = list(dict.fromkeys(normalized_sections))[:2]

            return final_output

        except Exception:

            # fallback: embedding-only routing
            return {
                q: await self._cosine_match(q)
                for q in queries
            }

In [11]:
query_batch = ['detail the terms of an automobile financing plan',
 'outline the conditions for obtaining a vehicle loan']

In [12]:
section_dict = await QueryClassifier().classify_batch(query_batch)

2026-04-18 13:42:05,958 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-18 13:42:05,960 INFO sqlalchemy.engine.Engine SELECT distinct(document_chunks.chunk_metadata ->> $1::TEXT) AS distinct_1 
FROM document_chunks 
WHERE document_chunks.chunk_metadata[$2::TEXT] IS NOT NULL
2026-04-18 13:42:05,961 INFO sqlalchemy.engine.Engine [cached since 4.845s ago] ('section_name', 'section_name')
[2026-04-18 13:42:05,966] [INFO] Extracted 7 normalized sections.
[2026-04-18 13:42:05,967] [INFO] get_normalized_sections completed in 0.02s
2026-04-18 13:42:05,971 INFO sqlalchemy.engine.Engine ROLLBACK


In [13]:
section_dict

{'detail the terms of an automobile financing plan': ['drivesure auto loan dal',
  'flexiserve personal loan fpl'],
 'outline the conditions for obtaining a vehicle loan': ['drivesure auto loan dal',
  'edugrow education loan']}

In [14]:
import faiss
import numpy as np
from collections import defaultdict
from sqlalchemy import select
from src.RAG.models import ChunkModel
from src.db.main import get_session
from src.Utils.logger_setup import setup_logger, current_logger, track_performance
from src.Utils.exception_handler import CustomException


logger = setup_logger("vectorstore")
current_logger.set(logger)


class VectorStore:

    def __init__(self, dim=768):
        self.dim = dim

        # section -> FAISS index
        self.indexes = {}

        # section -> {fid -> ChunkModel}
        self.chunk_store = {}

        self.section_service = SectionService()
        self.valid_sections = set()

    # -------------------------------------------------
    # LOAD CANONICAL SECTIONS
    # -------------------------------------------------
    async def load_sections(self):

        sections = await self.section_service.get_normalized_sections()
        self.valid_sections = set(sections)

        for sec in self.valid_sections:
            if sec not in self.indexes:
                self.indexes[sec] = faiss.IndexHNSWFlat(self.dim, 32)
                self.chunk_store[sec] = {}

    # -------------------------------------------------
    # BUILD VECTOR STORE FROM DB
    # -------------------------------------------------
    @track_performance
    async def add(self):

        try:
            if not self.valid_sections:
                await self.load_sections()

            grouped = defaultdict(list)

            # ---------------- DB FETCH ----------------
            async for session in get_session():

                stmt = select(ChunkModel)
                result = await session.execute(stmt)
                chunks = result.scalars().all()

                for c in chunks:

                    # ❗ SAFE embedding check (FIXED BUG)
                    if c.embedding is None:
                        logger.warning(f"Skipping chunk {c.id} (no embedding)")
                        continue

                    embedding = np.asarray(c.embedding, dtype="float32")

                    if embedding.size == 0:
                        logger.warning(f"Skipping chunk {c.id} (empty embedding)")
                        continue

                    section = normalize_section(
                        c.chunk_metadata.get("section_name")
                    )

                    if section not in self.valid_sections:
                        section = "default"

                    grouped[section].append((c, embedding))

                break

            # -------------------------------------------------
            # BUILD FAISS INDEX PER SECTION
            # -------------------------------------------------
            for section, items in grouped.items():

                index = self.indexes.get(section)

                if index is None:
                    index = faiss.IndexHNSWFlat(self.dim, 32)
                    self.indexes[section] = index
                    self.chunk_store[section] = {}

                vectors = np.array(
                    [emb for _, emb in items],
                    dtype="float32"
                )

                faiss.normalize_L2(vectors)

                start_id = index.ntotal
                index.add(vectors)

                for i, (chunk, _) in enumerate(items):
                    fid = start_id + i
                    self.chunk_store[section][fid] = chunk

            logger.info(
                f"VectorStore built successfully | sections={len(grouped)}"
            )

        except Exception as e:
            logger.error("Failed to build VectorStore from DB")
            raise CustomException(e)

    # -------------------------------------------------
    # SEARCH
    # -------------------------------------------------
    def search(self, query_vector, sections, k=5):

        query = np.asarray(query_vector, dtype="float32").reshape(1, -1)
        faiss.normalize_L2(query)

        results = []

        for sec in sections:

            sec = normalize_section(sec)

            index = self.indexes.get(sec)
            if not index:
                continue

            D, I = index.search(query, k)

            for score, idx in zip(D[0], I[0]):

                if idx == -1:
                    continue

                chunk = self.chunk_store[sec].get(idx)

                if chunk:
                    results.append({
                        "chunk": chunk,
                        "score": float(score)
                    })

        results.sort(key=lambda x: x["score"], reverse=True)
        return results[:k]

In [15]:
import asyncio

store = VectorStore(dim=768)

await store.add()

2026-04-18 13:43:12,955 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-18 13:43:12,961 INFO sqlalchemy.engine.Engine SELECT distinct(document_chunks.chunk_metadata ->> $1::TEXT) AS distinct_1 
FROM document_chunks 
WHERE document_chunks.chunk_metadata[$2::TEXT] IS NOT NULL
2026-04-18 13:43:12,966 INFO sqlalchemy.engine.Engine [cached since 71.85s ago] ('section_name', 'section_name')
[2026-04-18 13:43:13,072] [INFO] Extracted 7 normalized sections.
[2026-04-18 13:43:13,072] [INFO] get_normalized_sections completed in 0.19s
2026-04-18 13:43:13,115 INFO sqlalchemy.engine.Engine ROLLBACK
2026-04-18 13:43:13,411 INFO sqlalchemy.engine.Engine BEGIN (implicit)
2026-04-18 13:43:13,417 INFO sqlalchemy.engine.Engine SELECT document_chunks.id, document_chunks.chunk_text, document_chunks.embedding, document_chunks.confidence_score, document_chunks.chunk_metadata, document_chunks.prev_chunk_id, document_chunks.next_chunk_id, document_chunks.created_at, document_chunks.context_chunks 
FROM 

2026-04-18 13:43:14,286 INFO sqlalchemy.engine.Engine ROLLBACK


In [16]:
from src.RAG.models import ChunkModel
from src.RAG.Strategies_RAG_build.embeddings import DocumentEmbedder
import asyncio

async def run_flat_search(store, section_dict, k=10):

    all_results = []

    for query, sections in section_dict.items():

        # 1. embed query
        embedder = DocumentEmbedder()
        query_vector = await embedder.embed_query(query)

        # 2. search FAISS
        results = store.search(
            query_vector=query_vector,
            sections=sections,
            k=k
        )

        # 3. append in required format
        for r in results:
            all_results.append({
                "chunk": r["chunk"],   # ChunkModel
                "score": r["score"]
            })

    # optional: global sort across all queries
    all_results.sort(key=lambda x: x["score"], reverse=True)

    return all_results[:k]

In [17]:
sections_names

['drivesure auto loan dal',
 'edugrow education loan',
 'flexiserve personal loan fpl',
 'homesecure home loan hhl',
 'homesure mortgage loan hml',
 'securesave loan against approved securities slaas',
 'tradeplus business loan tbl']

In [18]:
section_dict

{'detail the terms of an automobile financing plan': ['drivesure auto loan dal',
  'flexiserve personal loan fpl'],
 'outline the conditions for obtaining a vehicle loan': ['drivesure auto loan dal',
  'edugrow education loan']}

In [19]:
from src.RAG.models import ChunkModel
from src.RAG.Strategies_RAG_build.embeddings import DocumentEmbedder
import asyncio

async def run_flat_search(store, section_dict, k=10):
    all_results = []
    seen_ids = set() # To track unique chunks
    embedder = DocumentEmbedder()

    for query, sections in section_dict.items():
        # 1. Use only the first entry of the section list (0 index)
        target_section = [sections[0]] if sections else []

        # 2. Embed query
        query_vector = await embedder.embed_query(query)

        # 3. Search FAISS
        query_results = store.search(
            query_vector=query_vector,
            sections=target_section,
            k=k
        )

        # 4. Filter and Deduplicate per query
        for r in query_results:
            chunk = r["chunk"]
            # Assuming ChunkModel has an 'id' or 'metadata["id"]' 
            # If no ID exists, use hash(chunk.page_content)
            chunk_id = getattr(chunk, 'id', chunk.id)

            if chunk_id not in seen_ids:
                all_results.append({
                    "chunk": chunk,
                    "score": r["score"]
                })
                seen_ids.add(chunk_id)

    # 5. Global sort and truncate
    all_results.sort(key=lambda x: x["score"], reverse=True)

    return all_results[:k]

In [20]:
results = await run_flat_search(
    store=store,
    section_dict= section_dict,
    k=10
)



In [21]:
results[1]

{'chunk': ChunkModel(embedding=array([ 5.03642023e-01,  5.32287300e-01, -2.88557458e+00,  4.89900231e-01,
         1.09165347e+00, -1.09649789e+00, -1.14245988e-01, -7.98029378e-02,
        -3.02703887e-01,  2.23931536e-01, -1.90546528e-01, -2.23284051e-01,
         1.12774706e+00, -7.24877059e-01,  2.46832207e-01,  1.25969589e-01,
        -5.24063349e-01, -1.60029101e+00,  4.60347205e-01,  7.10552454e-01,
        -9.33299601e-01, -2.41854578e-01, -1.17831326e+00, -5.00869118e-02,
         1.75065017e+00, -3.61719847e-01, -1.90472454e-01, -3.79411727e-01,
         3.24253023e-01,  2.52643436e-01,  1.54545918e-01, -7.17120767e-01,
         7.85444260e-01, -1.56673253e+00, -1.93407989e+00, -7.17504025e-01,
         1.09759617e+00,  1.07229781e+00,  2.60794818e-01, -1.35775602e+00,
         9.30494606e-01, -1.14860274e-01, -9.18065235e-02, -2.63932228e-01,
         1.62442136e+00, -6.25478268e-01,  2.09394956e+00, -1.10698715e-01,
         1.95695329e+00,  3.67204472e-02, -6.00571418e-03,

In [22]:
for sec, idx in store.indexes.items():
    print(sec, idx.ntotal)

homesecure home loan hhl 53
tradeplus business loan tbl 113
edugrow education loan 92
flexiserve personal loan fpl 78
drivesure auto loan dal 86
securesave loan against approved securities slaas 73
homesure mortgage loan hml 119


In [23]:
query_batch = ['detail the terms of an automobile financing plan',
 'outline the conditions for obtaining a vehicle loan']

In [24]:
import asyncio
from typing import List, Dict, Any
def extract_unique_texts(results_list: List[Dict[str, Any]]) -> List[str]:
        """
        Extracts primary chunk_text and all context_chunks text into a unique list.
        """
        all_texts = set()
        
        for item in results_list:
            chunk_obj = item.get("chunk")
            if not chunk_obj:
                continue

            # 1. Add the primary text
            if hasattr(chunk_obj, 'chunk_text'):
                all_texts.add(chunk_obj.chunk_text)

            # 2. Add text from context_chunks (list of dicts)
            context_chunks = getattr(chunk_obj, 'context_chunks', [])
            for ctx in context_chunks:
                if isinstance(ctx, dict) and 'chunk_text' in ctx:
                    all_texts.add(ctx['chunk_text'])
                elif hasattr(ctx, 'chunk_text'):
                    all_texts.add(ctx.chunk_text)

        unique_list = list(all_texts)
        print(f"Extraction Complete: Found {len(unique_list)} unique text segments.")
        return unique_list

In [25]:
final_results = extract_unique_texts(results_list=results)
final_results

Extraction Complete: Found 35 unique text segments.


['jeeps, and station wagons for private use; old four-wheelers not more than three years old ; new',
 '. The product enables borrowers to acquire personal vehicles with flexible repayment options while',
 'The loan may be availed for the purchase of new four-wheelers such as cars, jeeps, and station',
 '. Dealer advances are treated as margin only after due verification.',
 'Standard documentation includes Demand Promissory Note, hypothecation agreement, installment letter',
 'Total deductions including the proposed EMI shall not exceed 50% of gross monthly income for income',
 '. The product is intended exclusively for non-commercial, private use and provides structured',
 '. Vehicles financed under this scheme must not be registered or used as commercial vehicles.',
 'of the competent authority and approved deviation guidelines.',
 '. Gas kits require a margin of 15% , and two-wheelers require 10% margin on invoice value',
 'options while ensuring strong asset protection and regulato

In [30]:
from typing import List, Dict, Any
from sentence_transformers import CrossEncoder

from src.Utils.logger_setup import setup_logger, current_logger, track_performance

logger = setup_logger("reranker")
current_logger.set(logger)


class CrossEncoderReranker:
    def __init__(
        self,
        model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2",
        device: str = "cpu",
        batch_size: int = 8
    ):
        """
        Cross-encoder reranker (sync version, optimized with batching).
        """
        self.model_name = model_name
        self.batch_size = batch_size

        self.model = CrossEncoder(model_name, device=device)

        logger.info(
            f"Initialized CrossEncoderReranker | model={model_name} | device={device} | batch_size={batch_size}"
        )

    # -----------------------------
    # FAST UNIQUE TEXT EXTRACTION
    # -----------------------------
    def _extract_unique_texts(self, results_list: List[Dict[str, Any]]) -> List[str]:
        seen = set()
        flat_texts = []

        for item in results_list:
            chunk = item.get("chunk")
            if not chunk:
                continue

            text = getattr(chunk, "chunk_text", None)
            if text and text not in seen:
                seen.add(text)
                flat_texts.append(text)

            for ctx in getattr(chunk, "context_chunks", []) or []:
                ctx_text = (
                    ctx.get("chunk_text") if isinstance(ctx, dict)
                    else getattr(ctx, "chunk_text", None)
                )

                if ctx_text and ctx_text not in seen:
                    seen.add(ctx_text)
                    flat_texts.append(ctx_text)

        logger.info(f"Extracted {len(flat_texts)} unique texts")
        return flat_texts

    # -----------------------------
    # MAIN RERANK FUNCTION
    # -----------------------------
    @track_performance
    def rerank_chunks(
        self,
        queries: List[str],
        results_list: List[Dict[str, Any]],
        top_n: int = 10
    ) -> Dict[str, List[str]]:

        if not queries or not results_list:
            logger.warning("Empty queries or results_list provided.")
            return {}

        # Step 1: flatten texts
        flat_texts = self._extract_unique_texts(results_list)
        if not flat_texts:
            return {}

        # Step 2: merge query
        merged_query = " ".join(queries)

        # Step 3: create pairs
        pairs = [(merged_query, text) for text in flat_texts]

        # Step 4: inference
        scores = self.model.predict(
            pairs,
            batch_size=self.batch_size,
            show_progress_bar=False
        )

        # Step 5: rank
        ranked = sorted(
            zip(flat_texts, scores),
            key=lambda x: x[1],
            reverse=True
        )

        # Step 6: top N
        top_texts = [text for text, _ in ranked[:top_n]]

        logger.info(
            f"Reranked {len(flat_texts)} texts → returning top {top_n}"
        )

        # ✅ FINAL OUTPUT FORMAT
        return {
            merged_query: top_texts
        }

In [31]:
reranker = CrossEncoderReranker()
top_results = reranker.rerank_chunks(query_batch, results)
print(top_results)

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 2220.44it/s]
BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


[2026-04-18 13:47:23,200] [INFO] Initialized CrossEncoderReranker | model=cross-encoder/ms-marco-MiniLM-L-6-v2 | device=cpu | batch_size=8
[2026-04-18 13:47:23,200] [INFO] Extracted 35 unique texts
[2026-04-18 13:47:23,722] [INFO] Reranked 35 texts → returning top 10
[2026-04-18 13:47:23,726] [INFO] rerank_chunks completed in 0.52s
{'detail the terms of an automobile financing plan outline the conditions for obtaining a vehicle loan': ['DriveSure Auto Loan provides a structured, transparent, and risk-controlled vehicle financing', 'vehicle financing solution that balances customer affordability , operational discipline , and', '. Vehicles financed under this scheme must not be registered or used as commercial vehicles.', 'The loan may be availed for the purchase of new four-wheelers such as cars, jeeps, and station', 'for car loans, and 2% of loan amount (minimum ₹250) for two-wheeler loans, plus applicable taxes', '. The product enables borrowers to acquire personal vehicles with flex

In [33]:
from typing import Dict, List, AsyncGenerator
import asyncio
import ollama


class LLMGenerator:
    def __init__(self, model: str = "phi3.5"):
        self.model = model

    # -----------------------------
    # PROMPT BUILDER
    # -----------------------------
    def _build_prompt(self, query: str, contexts: List[str]) -> str:
        context_block = "\n\n".join(
            [f"[Doc {i+1}]\n{c}" for i, c in enumerate(contexts)]
        )

        return f"""
You are an expert retail banking loan advisor specializing in personal loans, car loans, home loans, interest rates, fees, and repayment terms.

Answer strictly using the provided context only.

If context is insufficient, say: "Not enough information in the provided documents."

Be concise, accurate, and customer-friendly.

User Question:
{query}

Context:
{context_block}

Output:
- Direct Answer
- Key Points
""".strip()

    # -----------------------------
    # ASYNC STREAMING GENERATION
    # -----------------------------
    async def stream_generate(
        self,
        rerank_output: Dict[str, List[str]]
    ) -> AsyncGenerator[Dict[str, str], None]:
        """
        Async generator yielding streaming responses per query.
        """

        loop = asyncio.get_event_loop()

        for query, contexts in rerank_output.items():

            prompt = self._build_prompt(query, contexts)

            print(f"\n\n===== Query: {query} =====\n")

            full_response = ""

            # Run ollama.chat in a thread (since it's blocking)
            stream = await loop.run_in_executor(
                None,
                lambda: ollama.chat(
                    model=self.model,
                    messages=[
                        {
                            "role": "system",
                            "content": "You are a precise and compliant retail banking loan advisor."
                        },
                        {
                            "role": "user",
                            "content": prompt
                        }
                    ],
                    stream=True
                )
            )

            # Iterate streamed tokens
            for chunk in stream:
                token = chunk["message"]["content"]
                full_response += token

                print(token, end="", flush=True)

                # yield partial updates (optional for UI streaming)
                await asyncio.sleep(0)

            yield {query: full_response}

In [36]:
import asyncio

async def main():
    generator = LLMGenerator(model="qwen2.5:7b")

    rerank_output = top_results  

    async for result in generator.stream_generate(rerank_output):
        print("\n\nFINAL OUTPUT:\n", result)




In [ ]:
await main()



===== Query: detail the terms of an automobile financing plan outline the conditions for obtaining a vehicle loan =====

Direct Answer:
DriveSure Auto Loan offers flexible terms for financing personal vehicle purchases. Here are the key conditions:

Key Points:
1. **Eligible Vehicles**: The loan can be used to purchase new four-wheelers

In [2]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

False
No GPU
